# Part 1: Data Assessment

## Objective

Examine the provided arXiv research papers, identify the available file types and metadata,
assess the quality of the data, and determine what information is usable for the RAG pipeline.

In [4]:
from pathlib import Path
import pandas as pd
from pypdf import PdfReader

cwd = Path.cwd()

if (cwd / "data").exists():
    data_dir = cwd / "data"
else:
    data_dir = cwd.parent / "data"

print("Data directory:", data_dir)

Data directory: /Users/janhhavibeley/Desktop/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data


In [5]:
files = sorted([
    file for file in data_dir.iterdir()
    if file.is_file() and file.name != ".gitkeep"
])

print("number of data files:", len(files))

for file in files:
    print(file.name)

number of data files: 5
2608.20316v1_Pandora's AI Model Routing Box Efficient Allocation with Costly Value Estimation.pdf
2608.20318v1_AI4AI-Bench Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Imp.pdf
2608.20319v1_Inducing Task Models from Computer-Use Traces.pdf
2608.20320v1_An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and We.pdf
2608.20331v1_G-CARL Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical R.pdf


In [6]:
file_info = []

for file in files:
    file_info.append({
        "filename": file.name,
        "file_type": file.suffix.lower(),
        "size_mb": round(file.stat().st_size / (1024 * 1024), 2)
    })

file_df = pd.DataFrame(file_info)
file_df

,filename,file_type,size_mb
0,2608.20316v1_Pandora's AI Model Routing Box Ef...,.pdf,0.59
1,2608.20318v1_AI4AI-Bench Benchmarking LLM Agen...,.pdf,1.48
2,2608.20319v1_Inducing Task Models from Compute...,.pdf,4.67
3,2608.20320v1_An Agentic Approach for Active Da...,.pdf,2.58
4,2608.20331v1_G-CARL Grounded Checklist-Aligned...,.pdf,3.63


In [7]:
paper_info = []

for file in files:
    if file.suffix.lower() == ".pdf":
        try:
            reader = PdfReader(file)
            metadata = reader.metadata

            pages_with_text = 0
            total_characters = 0

            for page in reader.pages:
                text = page.extract_text()

                if text and text.strip():
                    pages_with_text += 1
                    total_characters += len(text)

            paper_info.append({
                "filename": file.name,
                "title": metadata.get("/Title") if metadata else None,
                "author": metadata.get("/Author") if metadata else None,
                "num_pages": len(reader.pages),
                "pages_with_text": pages_with_text,
                "text_characters": total_characters,
                "readable": total_characters > 0
            })

        except Exception as e:
            paper_info.append({
                "filename": file.name,
                "title": None,
                "author": None,
                "num_pages": None,
                "pages_with_text": 0,
                "text_characters": 0,
                "readable": False
            })

papers_df = pd.DataFrame(paper_info)
papers_df

,filename,title,author,num_pages,pages_with_text,text_characters,readable
0,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,28,28,89597,True
1,2608.20318v1_AI4AI-Bench Benchmarking LLM Agen...,AI4AI-Bench: Benchmarking LLM Agents in Algori...,Yizhe Chi; Wenyi Li; Deyao Hong; Xiaoqiu Wang;...,19,19,76707,True
2,2608.20319v1_Inducing Task Models from Compute...,Inducing Task Models from Computer-Use Traces,Yucheng Jiang; Zora Zhiruo Wang; Ruishi Chen; ...,23,23,85873,True
3,2608.20320v1_An Agentic Approach for Active Da...,An Agentic Approach for Active Data Collection...,Narges Ahmadi; Yubo Jiao; Jônatas Augusto Manz...,25,25,73593,True
4,2608.20331v1_G-CARL Grounded Checklist-Aligned...,G-CARL: Grounded Checklist-Aligned Reward Lear...,Shiao Xie; Siyu Chen; Jianwei Lv; Bo Yuan; Yuj...,9,9,48596,True


In [8]:
papers_df.isnull().sum()

filename           0
title              0
author             0
num_pages          0
pages_with_text    0
text_characters    0
readable           0
dtype: int64

In [9]:
papers_df[[
    "filename",
    "title",
    "author",
    "num_pages",
    "pages_with_text",
    "text_characters",
    "readable"
]]

,filename,title,author,num_pages,pages_with_text,text_characters,readable
0,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,28,28,89597,True
1,2608.20318v1_AI4AI-Bench Benchmarking LLM Agen...,AI4AI-Bench: Benchmarking LLM Agents in Algori...,Yizhe Chi; Wenyi Li; Deyao Hong; Xiaoqiu Wang;...,19,19,76707,True
2,2608.20319v1_Inducing Task Models from Compute...,Inducing Task Models from Computer-Use Traces,Yucheng Jiang; Zora Zhiruo Wang; Ruishi Chen; ...,23,23,85873,True
3,2608.20320v1_An Agentic Approach for Active Da...,An Agentic Approach for Active Data Collection...,Narges Ahmadi; Yubo Jiao; Jônatas Augusto Manz...,25,25,73593,True
4,2608.20331v1_G-CARL Grounded Checklist-Aligned...,G-CARL: Grounded Checklist-Aligned Reward Lear...,Shiao Xie; Siyu Chen; Jianwei Lv; Bo Yuan; Yuj...,9,9,48596,True


In [10]:
for file in files:
    reader = PdfReader(file)
    metadata = reader.metadata

    print("\nFILE:", file.name)

    if metadata:
        print("Metadata fields:", list(metadata.keys()))
    else:
        print("No metadata found")


FILE: 2608.20316v1_Pandora's AI Model Routing Box Efficient Allocation with Costly Value Estimation.pdf
Metadata fields: ['/Author', '/Creator', '/DOI', '/License', '/PTEX.Fullbanner', '/Producer', '/Title', '/Trapped', '/arXivID']

FILE: 2608.20318v1_AI4AI-Bench Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Imp.pdf
Metadata fields: ['/Author', '/Creator', '/DOI', '/License', '/PTEX.Fullbanner', '/Producer', '/Title', '/Trapped', '/arXivID']

FILE: 2608.20319v1_Inducing Task Models from Computer-Use Traces.pdf
Metadata fields: ['/Author', '/Creator', '/DOI', '/License', '/PTEX.Fullbanner', '/Producer', '/Title', '/Trapped', '/arXivID']

FILE: 2608.20320v1_An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and We.pdf
Metadata fields: ['/Author', '/Creator', '/DOI', '/License', '/PTEX.Fullbanner', '/Producer', '/Title', '/Trapped', '/arXivID']

FILE: 2608.20331v1_G-CARL Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical R.pd

In [11]:
reader = PdfReader(files[0])
sample_text = reader.pages[0].extract_text()

print(sample_text[:2000])

2026-08-21
Pandora’s AI Model Routing Box: Efficient
Allocation with Costly Value Estimation
Adam Fisch∗, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate
Larson, Jacob Eisenstein∗
Google DeepMind
Heterogeneous AI systems composed of multiple models, architectures, harnesses, or inference-time
settings can improve quality and efficiency by routing queries to thespecialistwho can answer most
effectively at the lowest cost. Routing requires estimating each specialist’s expected return, but this value
estimation has a cost. Cheap estimators (e.g., embedding-based predictors) are fast but noisy, while ac-
curate estimators (e.g., fine-tuned models with access to retrieval results or partial reasoning traces) are
expensive. We formalize this tradeoff as an instance of Pandora’s Box, the classical problem of optimal
search with costly inspection. Under a Gaussian signal model, the resulting policies have closed-form
value-of-information expressions tha

In [12]:
for file in files:
    reader = PdfReader(file)
    text = reader.pages[0].extract_text()

    print("\nFILE:", file.name)
    print(text[:300])
    print("-" * 80)


FILE: 2608.20316v1_Pandora's AI Model Routing Box Efficient Allocation with Costly Value Estimation.pdf
2026-08-21
Pandora’s AI Model Routing Box: Efficient
Allocation with Costly Value Estimation
Adam Fisch∗, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate
Larson, Jacob Eisenstein∗
Google DeepMind
Heterogeneous AI systems composed of multiple models, architect
--------------------------------------------------------------------------------

FILE: 2608.20318v1_AI4AI-Bench Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Imp.pdf
AI4AI-Bench: Benchmarking LLM Agents in
Algorithmic Design for Recursive Self-Improvement
Yizhe Chi †, Wenyi Li, Deyao Hong, Xiaoqiu Wang, Mingju Gao, Kaisen Yang, Bingxiang He, Y oujie Zheng, Calvin Xiao,
Qinhuai Na ‡
Navers Lab, Einsia.AI T singhua University
†Project Lead ‡Corresponding Author
Ab
--------------------------------------------------------------------------------

FILE: 2608.20319v1_Induci

In [13]:
print("duplicate filenames:", file_df["filename"].duplicated().sum())
print("duplicate titles:", papers_df["title"].duplicated().sum())

duplicate filenames: 0
duplicate titles: 0


In [14]:
metadata_rows = []

for file in files:
    reader = PdfReader(file)
    metadata = reader.metadata

    metadata_rows.append({
        "filename": file.name,
        "title": metadata.get("/Title"),
        "author": metadata.get("/Author"),
        "doi": metadata.get("/DOI"),
        "license": metadata.get("/License")
    })

metadata_df = pd.DataFrame(metadata_rows)
metadata_df

,filename,title,author,doi,license
0,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,https://doi.org/10.48550/arXiv.2608.20316,http://creativecommons.org/licenses/by/4.0/
1,2608.20318v1_AI4AI-Bench Benchmarking LLM Agen...,AI4AI-Bench: Benchmarking LLM Agents in Algori...,Yizhe Chi; Wenyi Li; Deyao Hong; Xiaoqiu Wang;...,https://doi.org/10.48550/arXiv.2608.20318,http://creativecommons.org/licenses/by/4.0/
2,2608.20319v1_Inducing Task Models from Compute...,Inducing Task Models from Computer-Use Traces,Yucheng Jiang; Zora Zhiruo Wang; Ruishi Chen; ...,https://doi.org/10.48550/arXiv.2608.20319,http://arxiv.org/licenses/nonexclusive-distrib...
3,2608.20320v1_An Agentic Approach for Active Da...,An Agentic Approach for Active Data Collection...,Narges Ahmadi; Yubo Jiao; Jônatas Augusto Manz...,https://doi.org/10.48550/arXiv.2608.20320,http://arxiv.org/licenses/nonexclusive-distrib...
4,2608.20331v1_G-CARL Grounded Checklist-Aligned...,G-CARL: Grounded Checklist-Aligned Reward Lear...,Shiao Xie; Siyu Chen; Jianwei Lv; Bo Yuan; Yuj...,https://doi.org/10.48550/arXiv.2608.20331,http://arxiv.org/licenses/nonexclusive-distrib...


## Data Quality Findings

- The dataset contains 5 unique arXiv research papers, all in PDF format.
- All 5 PDFs opened successfully and contained extractable text on every page.
- Titles and author information were available for all papers.
- DOI and license information were also available in the PDF metadata.
- No duplicate filenames or paper titles were found.
- No missing values were found in the fields collected during the assessment.
- The extracted text is readable overall, although some formatting artifacts are present, including missing spaces, merged words, and extra symbols around author names and affiliations.



### Usability for the RAG Pipeline

The following information should be useful for the RAG pipeline:
- Full paper text
- Paper titles
- Author information
- arXiv IDs from the filenames
- DOI and license information
- PDF page information that can be preserved later for citations and source tracking

The dataset looks usable for the RAG prototype. All 5 papers were readable, no duplicate filenames or titles were found, and the metadata needed for source tracking is available. Some formatting issues were found in the extracted text, including missing spaces, merged words, and extra symbols around author names and affiliations. These issues should be handled during the data cleaning and preprocessing stage before the text is split into chunks for retrieval.

# Part 2: Data Prep

Task #3 clean and process data 

In [15]:
# extra imports , output dir

import re
import json
import unicodedata
from dataclasses import dataclass
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("ingestion_pipeline")

processed_dir = data_dir.parent / "data_processed"
processed_dir.mkdir(exist_ok=True)

print("Processed output directory:", processed_dir)

Processed output directory: /Users/janhhavibeley/Desktop/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data_processed


In [16]:
# extract per page text

def extract_arxiv_id(filename: str) -> str:
    """Pull the arXiv ID (e.g. 2608.20316v1) off the front of the filename."""
    match = re.match(r"([\d]{4}\.[\d]{5}v\d+)", filename)
    return match.group(1) if match else filename


def extract_pdf_pages(file_path: Path) -> list[dict]:
    """Extract one record per page, with document-level metadata attached to every row."""
    reader = PdfReader(file_path)
    meta = reader.metadata or {}

    records = []
    for page_number, page in enumerate(reader.pages, start=1):
        raw_text = page.extract_text() or ""
        records.append({
            "arxiv_id": extract_arxiv_id(file_path.name),
            "filename": file_path.name,
            "title": meta.get("/Title"),
            "author": meta.get("/Author"),
            "doi": meta.get("/DOI"),
            "license": meta.get("/License"),
            "page_number": page_number,
            "num_pages": len(reader.pages),
            "raw_text": raw_text
        })
    return records


all_pages = []
for file in files:
    logger.info(f"Extracting {file.name}")
    all_pages.extend(extract_pdf_pages(file))

pages_df = pd.DataFrame(all_pages)
print(f"Extracted {len(pages_df)} pages from {pages_df['filename'].nunique()} papers")
pages_df.head()

2026-10-06 15:33:52,315 | INFO | Extracting 2608.20316v1_Pandora's AI Model Routing Box Efficient Allocation with Costly Value Estimation.pdf
2026-10-06 15:33:55,251 | INFO | Extracting 2608.20318v1_AI4AI-Bench Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Imp.pdf
2026-10-06 15:33:58,389 | INFO | Extracting 2608.20319v1_Inducing Task Models from Computer-Use Traces.pdf
2026-10-06 15:34:00,092 | INFO | Extracting 2608.20320v1_An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and We.pdf
2026-10-06 15:34:01,970 | INFO | Extracting 2608.20331v1_G-CARL Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical R.pdf


Extracted 104 pages from 5 papers


,arxiv_id,filename,title,author,doi,license,page_number,num_pages,raw_text
0,2608.20316v1,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,https://doi.org/10.48550/arXiv.2608.20316,http://creativecommons.org/licenses/by/4.0/,1,28,2026-08-21\nPandora’s AI Model Routing Box: Ef...
1,2608.20316v1,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,https://doi.org/10.48550/arXiv.2608.20316,http://creativecommons.org/licenses/by/4.0/,2,28,Pandora’s AI Model Routing Box: Efficient Allo...
2,2608.20316v1,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,https://doi.org/10.48550/arXiv.2608.20316,http://creativecommons.org/licenses/by/4.0/,3,28,Pandora’s AI Model Routing Box: Efficient Allo...
3,2608.20316v1,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,https://doi.org/10.48550/arXiv.2608.20316,http://creativecommons.org/licenses/by/4.0/,4,28,Pandora’s AI Model Routing Box: Efficient Allo...
4,2608.20316v1,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,https://doi.org/10.48550/arXiv.2608.20316,http://creativecommons.org/licenses/by/4.0/,5,28,Pandora’s AI Model Routing Box: Efficient Allo...


In [17]:
# clean text

def clean_text(text: str) -> str:
    """Clean raw PDF-extracted text based on artifacts identified in the Part 1 assessment."""
    if not text or not text.strip():
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"-\s*\n\s*", "", text)              # rejoin hyphenated line breaks
    text = re.sub(r"(\w)-\s+(\w)", r"\1-\2", text)      # rejoin hyphenated stray spacing
    text = re.sub(r"\s[\*†‡]\s", " ", text)             # strip isolated footnote markers
    text = re.sub(r"\n+", " ", text)                    # collapse newlines
    text = re.sub(r"[ \t]+", " ", text)                 # collapse repeated whitespace

    return text.strip()


pages_df["cleaned_text"] = pages_df["raw_text"].apply(clean_text)
pages_df["char_count"] = pages_df["cleaned_text"].str.len()
pages_df["word_count"] = pages_df["cleaned_text"].str.split().str.len()

print("BEFORE:\n", pages_df.loc[0, "raw_text"][:300])
print("\nAFTER:\n", pages_df.loc[0, "cleaned_text"][:300])

BEFORE:
 2026-08-21
Pandora’s AI Model Routing Box: Efficient
Allocation with Costly Value Estimation
Adam Fisch∗, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate
Larson, Jacob Eisenstein∗
Google DeepMind
Heterogeneous AI systems composed of multiple models, architect

AFTER:
 2026-08-21 Pandora’s AI Model Routing Box: Efficient Allocation with Costly Value Estimation Adam Fisch∗, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate Larson, Jacob Eisenstein∗ Google DeepMind Heterogeneous AI systems composed of multiple models, architect


In [18]:
# chunk text

CHUNK_SIZE_WORDS = 250
CHUNK_OVERLAP_WORDS = 40


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE_WORDS, overlap: int = CHUNK_OVERLAP_WORDS) -> list[str]:
    """Split text into overlapping word-based chunks. Returns [] for empty text."""
    words = text.split()
    if not words:
        return []
    if len(words) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks


chunk_records = []
for _, row in pages_df.iterrows():
    page_chunks = chunk_text(row["cleaned_text"])
    for chunk_index, chunk in enumerate(page_chunks):
        chunk_records.append({
            "chunk_id": f"{row['arxiv_id']}_p{row['page_number']}_c{chunk_index}",
            "arxiv_id": row["arxiv_id"],
            "filename": row["filename"],
            "title": row["title"],
            "author": row["author"],
            "doi": row["doi"],
            "license": row["license"],
            "page_number": row["page_number"],
            "chunk_index": chunk_index,
            "text": chunk,
            "word_count": len(chunk.split())
        })

chunks_df = pd.DataFrame(chunk_records)
print(f"Produced {len(chunks_df)} chunks from {pages_df['filename'].nunique()} papers")
chunks_df[["chunk_id", "page_number", "word_count"]].head(10)

Produced 300 chunks from 5 papers


,chunk_id,page_number,word_count
0,2608.20316v1_p1_c0,1,250
1,2608.20316v1_p1_c1,1,250
2,2608.20316v1_p1_c2,1,124
3,2608.20316v1_p2_c0,2,250
4,2608.20316v1_p2_c1,2,250
5,2608.20316v1_p2_c2,2,204
6,2608.20316v1_p3_c0,3,250
7,2608.20316v1_p3_c1,3,250
8,2608.20316v1_p3_c2,3,244
9,2608.20316v1_p3_c3,3,34


In [19]:
# validate

assert not chunks_df.empty, "No chunks were produced"
assert chunks_df["chunk_id"].is_unique, "Duplicate chunk IDs found"
assert (chunks_df["word_count"] > 0).all(), "Empty chunks found"
assert chunks_df["title"].notna().all(), "Chunks missing title metadata"
assert chunks_df["arxiv_id"].notna().all(), "Chunks missing arxiv_id"
assert (chunks_df["word_count"] <= CHUNK_SIZE_WORDS + 5).all(), "Chunk exceeds target size unexpectedly"

print("All validation checks passed.")
print("\nChunk word-count distribution:")
print(chunks_df["word_count"].describe())

All validation checks passed.

Chunk word-count distribution:
count    300.000000
mean     201.453333
std       73.333716
min        2.000000
25%      149.500000
50%      250.000000
75%      250.000000
max      250.000000
Name: word_count, dtype: float64


In [20]:
#save 
#install pyarrow

%pip install pyarrow

jsonl_path = processed_dir / "chunks.jsonl"
parquet_path = processed_dir / "chunks.parquet"

with open(jsonl_path, "w") as f:
    for record in chunks_df.to_dict(orient="records"):
        f.write(json.dumps(record) + "\n")

chunks_df.to_parquet(parquet_path, index=False)

logger.info(f"Saved {len(chunks_df)} chunks to {jsonl_path}")
logger.info(f"Saved {len(chunks_df)} chunks to {parquet_path}")

Note: you may need to restart the kernel to use updated packages.


2026-10-06 15:34:05,770 | INFO | Saved 300 chunks to /Users/janhhavibeley/Desktop/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data_processed/chunks.jsonl
2026-10-06 15:34:05,771 | INFO | Saved 300 chunks to /Users/janhhavibeley/Desktop/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data_processed/chunks.parquet


In [21]:
# wrap as a repeatable pipeline 

@dataclass
class IngestionConfig:
    data_dir: Path
    output_dir: Path
    chunk_size_words: int = CHUNK_SIZE_WORDS
    chunk_overlap_words: int = CHUNK_OVERLAP_WORDS


class ArxivIngestionPipeline:
    def __init__(self, config: IngestionConfig):
        self.config = config
        self.pages_df = None
        self.chunks_df = None

    def extract(self):
        pdf_files = sorted([f for f in self.config.data_dir.iterdir() if f.suffix.lower() == ".pdf"])
        records = []
        for file in pdf_files:
            logger.info(f"Extracting {file.name}")
            records.extend(extract_pdf_pages(file))
        self.pages_df = pd.DataFrame(records)
        return self

    def clean(self):
        self.pages_df["cleaned_text"] = self.pages_df["raw_text"].apply(clean_text)
        return self

    def chunk(self):
        records = []
        for _, row in self.pages_df.iterrows():
            page_chunks = chunk_text(
                row["cleaned_text"],
                chunk_size=self.config.chunk_size_words,
                overlap=self.config.chunk_overlap_words
            )
            for chunk_index, chunk in enumerate(page_chunks):
                records.append({
                    "chunk_id": f"{row['arxiv_id']}_p{row['page_number']}_c{chunk_index}",
                    "arxiv_id": row["arxiv_id"],
                    "filename": row["filename"],
                    "title": row["title"],
                    "author": row["author"],
                    "doi": row["doi"],
                    "license": row["license"],
                    "page_number": row["page_number"],
                    "chunk_index": chunk_index,
                    "text": chunk,
                    "word_count": len(chunk.split())
                })
        self.chunks_df = pd.DataFrame(records)
        return self

    def validate(self):
        assert not self.chunks_df.empty, "No chunks produced"
        assert self.chunks_df["chunk_id"].is_unique, "Duplicate chunk IDs"
        assert (self.chunks_df["word_count"] > 0).all(), "Empty chunks found"
        assert self.chunks_df["title"].notna().all(), "Missing title metadata"
        return self

    def load(self):
        self.config.output_dir.mkdir(exist_ok=True)
        jsonl_path = self.config.output_dir / "chunks.jsonl"
        with open(jsonl_path, "w") as f:
            for record in self.chunks_df.to_dict(orient="records"):
                f.write(json.dumps(record) + "\n")
        self.chunks_df.to_parquet(self.config.output_dir / "chunks.parquet", index=False)
        logger.info(f"Saved {len(self.chunks_df)} chunks to {self.config.output_dir}")
        return self

    def run(self):
        return self.extract().clean().chunk().validate().load()


config = IngestionConfig(data_dir=data_dir, output_dir=processed_dir)
pipeline = ArxivIngestionPipeline(config)
pipeline.run()
print(f"Pipeline complete: {len(pipeline.chunks_df)} chunks ready for embedding")

2026-10-06 15:34:05,790 | INFO | Extracting 2608.20316v1_Pandora's AI Model Routing Box Efficient Allocation with Costly Value Estimation.pdf
2026-10-06 15:34:07,923 | INFO | Extracting 2608.20318v1_AI4AI-Bench Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Imp.pdf
2026-10-06 15:34:09,796 | INFO | Extracting 2608.20319v1_Inducing Task Models from Computer-Use Traces.pdf
2026-10-06 15:34:11,404 | INFO | Extracting 2608.20320v1_An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and We.pdf
2026-10-06 15:34:12,678 | INFO | Extracting 2608.20331v1_G-CARL Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical R.pdf
2026-10-06 15:34:13,735 | INFO | Saved 300 chunks to /Users/janhhavibeley/Desktop/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data_processed


Pipeline complete: 300 chunks ready for embedding


## Summary

- Extracted per-page text and metadata for all 5 papers.
- Cleaned hyphenation breaks, stray footnote symbols, and irregular whitespace found during Part 1.
- Chunked text at the page level (250 words, 40-word overlap) so every chunk keeps an accurate page number for citations.
- Validated that every chunk has complete metadata, a unique ID, and non-empty text.
- Packaged the flow into `ArxivIngestionPipeline` for repeatable reruns.
- Output: `data_processed/chunks.jsonl` and `chunks.parquet`, ready for embedding.

# Part 3

RAG Model building

In [22]:
# install dependencies
# llamaindex is the core framework that wraps the other components (embeddings, vector stores, LLMs) together
# chromadb is a vector database for the embeddings

%pip install llama-index-core llama-index-embeddings-huggingface llama-index-vector-stores-chroma llama-index-llms-groq chromadb

import os
import chromadb

from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.core.schema import TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.llms.groq import Groq

Note: you may need to restart the kernel to use updated packages.


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
# load chunks

chunks_df = pd.read_parquet(processed_dir / "chunks.parquet")
print(f"loaded {len(chunks_df)} chunks from {processed_dir / 'chunks.parquet'}")

loaded 300 chunks from /Users/janhhavibeley/Desktop/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data_processed/chunks.parquet


In [24]:
# convert raw chunks into llamaindex textnodes, i.e. units with text and metadata

nodes = []
for _, row in chunks_df.iterrows():
    node = TextNode(
        text=row["text"],
        metadata={
            "arxiv_id": row["arxiv_id"],
            "filename": row["filename"],
            "title": row["title"],
            "author": row["author"],
            "doi": row["doi"],
            "license": row["license"],
            "page_number": row["page_number"],
        }
    )
    nodes.append(node)

print(f"built {len(nodes)} nodes")

built 300 nodes


In [25]:
# config embedding model

Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

2026-10-06 15:34:39,656 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 15:34:39,675 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-10-06 15:34:39,714 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 15:34:39,726 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-10-06 15:34:39,731 | INFO | Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-10-06 15:34:39,765 | INFO | HTTP Request: HEAD https://huggingface.co/s

# Part 4 - Baseline RAG Evaluation


In [26]:
# Part 4: Baseline RAG Evaluation

import chromadb
import pandas as pd

from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore

In [27]:
# Create benchmark questions

benchmark_questions = [
    {
        "id": "Q1",
        "question": "What problem does Pandora's AI Model Routing Box aim to solve?",
        "expected_arxiv_id": "2608.20316v1"
    },
    {
        "id": "Q2",
        "question": "How does Pandora's AI Model Routing Box improve the efficiency of AI model selection?",
        "expected_arxiv_id": "2608.20316v1"
    },
    {
        "id": "Q3",
        "question": "What is AI4AI-Bench designed to evaluate?",
        "expected_arxiv_id": "2608.20318v1"
    },
    {
        "id": "Q4",
        "question": "How does AI4AI-Bench assess the performance of LLM agents in algorithmic design?",
        "expected_arxiv_id": "2608.20318v1"
    },
    {
        "id": "Q5",
        "question": "What information is used to induce task models from computer-use traces?",
        "expected_arxiv_id": "2608.20319v1"
    },
    {
        "id": "Q6",
        "question": "What is the purpose of the agentic approach for active data collection and travel behaviour modelling?",
        "expected_arxiv_id": "2608.20320v1"
    },
    {
        "id": "Q7",
        "question": "What problem does grounded checklist-aligned reward learning address in patient-oriented medical research?",
        "expected_arxiv_id": "2608.20331v1"
    },
    {
        "id": "Q8",
        "question": "How is reward learning used to improve patient-oriented medical outcomes?",
        "expected_arxiv_id": "2608.20331v1"
    }
]

benchmark_df = pd.DataFrame(benchmark_questions)

benchmark_df

,id,question,expected_arxiv_id
0,Q1,What problem does Pandora's AI Model Routing B...,2608.20316v1
1,Q2,How does Pandora's AI Model Routing Box improv...,2608.20316v1
2,Q3,What is AI4AI-Bench designed to evaluate?,2608.20318v1
3,Q4,How does AI4AI-Bench assess the performance of...,2608.20318v1
4,Q5,What information is used to induce task models...,2608.20319v1
5,Q6,What is the purpose of the agentic approach fo...,2608.20320v1
6,Q7,What problem does grounded checklist-aligned r...,2608.20331v1
7,Q8,How is reward learning used to improve patient...,2608.20331v1


In [28]:
# Create Chroma vector store

chroma_client = chromadb.PersistentClient(
    path=str(processed_dir / "chroma_db")
)

chroma_collection = chroma_client.get_or_create_collection(
    name="baseline_rag"
)

vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

In [29]:
# Create the baseline vector index

index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    show_progress=True
)

retriever = index.as_retriever(
    similarity_top_k=5
)

Generating embeddings: 100%|██████████| 300/300 [00:45<00:00,  6.63it/s]


In [30]:
# Test baseline retrieval

test_question = benchmark_questions[0]["question"]

results = retriever.retrieve(test_question)

for rank, result in enumerate(results, start=1):
    print(f"Rank {rank}")
    print(f"arXiv ID: {result.node.metadata.get('arxiv_id')}")
    print(f"Title: {result.node.metadata.get('title')}")
    print(f"Page: {result.node.metadata.get('page_number')}")
    print()

Rank 1
arXiv ID: 2608.20316v1
Title: Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation
Page: 26

Rank 2
arXiv ID: 2608.20316v1
Title: Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation
Page: 26

Rank 3
arXiv ID: 2608.20316v1
Title: Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation
Page: 26

Rank 4
arXiv ID: 2608.20316v1
Title: Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation
Page: 18

Rank 5
arXiv ID: 2608.20316v1
Title: Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation
Page: 18



In [31]:
# Evaluate baseline retrieval

retrieval_results = []

for item in benchmark_questions:

    results = retriever.retrieve(item["question"])

    retrieved_ids = [
        result.node.metadata.get("arxiv_id")
        for result in results
    ]

    expected_id = item["expected_arxiv_id"]

    retrieval_results.append({
        "id": item["id"],
        "question": item["question"],
        "expected_arxiv_id": expected_id,
        "retrieved_ids": retrieved_ids,
        "hit_at_1": expected_id in retrieved_ids[:1],
        "hit_at_3": expected_id in retrieved_ids[:3],
        "hit_at_5": expected_id in retrieved_ids[:5]
    })

evaluation_df = pd.DataFrame(retrieval_results)

evaluation_df

,id,question,expected_arxiv_id,retrieved_ids,hit_at_1,hit_at_3,hit_at_5
0,Q1,What problem does Pandora's AI Model Routing B...,2608.20316v1,"[2608.20316v1, 2608.20316v1, 2608.20316v1, 260...",True,True,True
1,Q2,How does Pandora's AI Model Routing Box improv...,2608.20316v1,"[2608.20316v1, 2608.20316v1, 2608.20316v1, 260...",True,True,True
2,Q3,What is AI4AI-Bench designed to evaluate?,2608.20318v1,"[2608.20318v1, 2608.20318v1, 2608.20318v1, 260...",True,True,True
3,Q4,How does AI4AI-Bench assess the performance of...,2608.20318v1,"[2608.20318v1, 2608.20318v1, 2608.20318v1, 260...",True,True,True
4,Q5,What information is used to induce task models...,2608.20319v1,"[2608.20319v1, 2608.20319v1, 2608.20319v1, 260...",True,True,True
5,Q6,What is the purpose of the agentic approach fo...,2608.20320v1,"[2608.20320v1, 2608.20320v1, 2608.20320v1, 260...",True,True,True
6,Q7,What problem does grounded checklist-aligned r...,2608.20331v1,"[2608.20331v1, 2608.20331v1, 2608.20331v1, 260...",True,True,True
7,Q8,How is reward learning used to improve patient...,2608.20331v1,"[2608.20331v1, 2608.20331v1, 2608.20331v1, 260...",True,True,True


In [32]:
# Calculate retrieval metrics

hit_at_1 = evaluation_df["hit_at_1"].mean()
hit_at_3 = evaluation_df["hit_at_3"].mean()
hit_at_5 = evaluation_df["hit_at_5"].mean()

print("Baseline Retrieval Performance")
print("-" * 40)
print(f"Hit@1: {hit_at_1:.2%}")
print(f"Hit@3: {hit_at_3:.2%}")
print(f"Hit@5: {hit_at_5:.2%}")

Baseline Retrieval Performance
----------------------------------------
Hit@1: 100.00%
Hit@3: 100.00%
Hit@5: 100.00%


In [33]:
# Save baseline retrieval results

evaluation_df.to_csv(
    processed_dir / "baseline_retrieval_evaluation.csv",
    index=False
)

Okay, so far 100% Hit for 1,3, and 5 indicates that it was able to retrieve a chunk from the expected paper. However, we do not know if the RAG system generates correct answers.

Will be using Groq generation and evalluate the answers below.


In [34]:
# Set up Groq

import os
from dotenv import load_dotenv

from llama_index.core import Settings
from llama_index.llms.groq import Groq

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm = Groq(
    model="openai/gpt-oss-20b",
    api_key=groq_api_key
)

Settings.llm = llm

In [35]:
# Test Groq with one question

question = benchmark_questions[0]["question"]

results = retriever.retrieve(question)

context = "\n\n".join([
    f"""
Source:
Title: {result.node.metadata.get('title')}
arXiv ID: {result.node.metadata.get('arxiv_id')}
Page: {result.node.metadata.get('page_number')}

Text:
{result.node.text}
"""
    for result in results
])

prompt = f"""
You are a research intelligence assistant.

Answer the question using only the research context provided below.

Question:
{question}

Research context:
{context}

Instructions:
- Give a concise and factual answer.
- Do not use information outside the provided context.
- If the context does not contain enough information, say so.
- At the end of your answer, provide the source arXiv ID.
"""

response = llm.complete(prompt)

print(response.text)

2026-10-06 15:35:47,022 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Pandora’s AI Model Routing Box is designed to address the problem of efficiently allocating AI models to incoming tasks while taking into account the cost of estimating each model’s value. It seeks to minimize regret and inspection cost by routing queries to the most appropriate model under a budget‑constrained value‑estimation setting.  

Source: arXiv 2608.20316v1


In [36]:
# Generate a RAG answer

def generate_rag_answer(question):

    results = retriever.retrieve(question)

    context = "\n\n".join([
        f"""
Source:
Title: {result.node.metadata.get('title')}
arXiv ID: {result.node.metadata.get('arxiv_id')}
Page: {result.node.metadata.get('page_number')}

Text:
{result.node.text}
"""
        for result in results
    ])

    prompt = f"""
You are a research intelligence assistant.

Answer the question using only the research context provided below.

Question:
{question}

Research context:
{context}

Instructions:
- Give a concise and factual answer.
- Do not use information outside the provided context.
- If the context does not contain enough information, say so.
- At the end of your answer, provide the source arXiv ID.
"""

    response = llm.complete(prompt)

    return response.text, results

In [37]:
# Generate answers for all benchmark questions

rag_results = []

for item in benchmark_questions:

    answer, results = generate_rag_answer(
        item["question"]
    )

    retrieved_ids = [
        result.node.metadata.get("arxiv_id")
        for result in results
    ]

    rag_results.append({
        "id": item["id"],
        "question": item["question"],
        "expected_arxiv_id": item["expected_arxiv_id"],
        "answer": answer,
        "retrieved_ids": retrieved_ids
    })

rag_results_df = pd.DataFrame(rag_results)

rag_results_df

2026-10-06 15:35:48,711 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 15:35:50,150 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 15:35:50,305 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-10-06 15:35:50,308 | INFO | Retrying request to /chat/completions in 2.000000 seconds
2026-10-06 15:35:53,302 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 15:35:53,517 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-10-06 15:35:53,518 | INFO | Retrying request to /chat/completions in 17.000000 seconds
2026-10-06 15:36:11,728 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 15:36:13,087 | INFO | HTTP Request: POST https://api.groq.com/openai

,id,question,expected_arxiv_id,answer,retrieved_ids
0,Q1,What problem does Pandora's AI Model Routing B...,2608.20316v1,Pandora’s AI Model Routing Box tackles the cha...,"[2608.20316v1, 2608.20316v1, 2608.20316v1, 260..."
1,Q2,How does Pandora's AI Model Routing Box improv...,2608.20316v1,Pandora’s AI Model Routing Box improves effici...,"[2608.20316v1, 2608.20316v1, 2608.20316v1, 260..."
2,Q3,What is AI4AI-Bench designed to evaluate?,2608.20318v1,AI4AI‑Bench is a benchmark that tests whether ...,"[2608.20318v1, 2608.20318v1, 2608.20318v1, 260..."
3,Q4,How does AI4AI-Bench assess the performance of...,2608.20318v1,AI4AI‑Bench evaluates an LLM agent by letting ...,"[2608.20318v1, 2608.20318v1, 2608.20318v1, 260..."
4,Q5,What information is used to induce task models...,2608.20319v1,Task models are induced from the low‑level eve...,"[2608.20319v1, 2608.20319v1, 2608.20319v1, 260..."
5,Q6,What is the purpose of the agentic approach fo...,2608.20320v1,The agentic approach is designed to unify the ...,"[2608.20320v1, 2608.20320v1, 2608.20320v1, 260..."
6,Q7,What problem does grounded checklist-aligned r...,2608.20331v1,Grounded checklist‑aligned reward learning (G‑...,"[2608.20331v1, 2608.20331v1, 2608.20331v1, 260..."
7,Q8,How is reward learning used to improve patient...,2608.20331v1,Reward learning in G‑CARL improves patient‑ori...,"[2608.20331v1, 2608.20331v1, 2608.20331v1, 260..."


In [38]:
# Review generated answers

for _, row in rag_results_df.iterrows():

    print(f"--- {row['id']} ---")
    print(f"Question: {row['question']}")
    print()
    print(f"Answer: {row['answer']}")
    print()
    print(f"Expected source: {row['expected_arxiv_id']}")
    print("-" * 60)

--- Q1 ---
Question: What problem does Pandora's AI Model Routing Box aim to solve?

Answer: Pandora’s AI Model Routing Box tackles the challenge of **efficiently allocating tasks to AI models when estimating each model’s value is expensive**—i.e., it seeks to optimize routing decisions while accounting for the cost of value estimation.  

Source: arXiv 2608.20316v1

Expected source: 2608.20316v1
------------------------------------------------------------
--- Q2 ---
Question: How does Pandora's AI Model Routing Box improve the efficiency of AI model selection?

Answer: Pandora’s AI Model Routing Box improves efficiency by routing queries to the most valuable models using a cost‑aware value‑estimation framework.  The system employs value estimators (Section C.1) and carefully designed prompts (Section C.2) to predict each model’s expected benefit, allowing it to limit the number of inspections (see Supplemental Results D.1) and the monetary cost of value‑function evaluations (Supplemen

In [40]:
# Create answer evaluation table

answer_evaluation_df = rag_results_df[
    [
        "id",
        "question",
        "answer",
        "expected_arxiv_id"
    ]
].copy()

# Manual evaluation
# 0 = poor, 1 = partial, 2 = strong

answer_evaluation_df["answer_relevance"] = [
    2,  # Q1
    2,  # Q2
    2,  # Q3
    2,  # Q4
    2,  # Q5
    2,  # Q6
    2,  # Q7
    2   # Q8
]

answer_evaluation_df["faithfulness"] = [
    2,  # Q1
    2,  # Q2
    2,  # Q3
    2,  # Q4
    2,  # Q5
    2,  # Q6
    2,  # Q7
    2   # Q8
]

answer_evaluation_df["citation_accuracy"] = [
    2,  # Q1
    2,  # Q2
    2,  # Q3
    2,  # Q4
    2,  # Q5
    2,  # Q6
    2,  # Q7
    2   # Q8
]

answer_evaluation_df

,id,question,answer,expected_arxiv_id,answer_relevance,faithfulness,citation_accuracy
0,Q1,What problem does Pandora's AI Model Routing B...,Pandora’s AI Model Routing Box tackles the cha...,2608.20316v1,2,2,2
1,Q2,How does Pandora's AI Model Routing Box improv...,Pandora’s AI Model Routing Box improves effici...,2608.20316v1,2,2,2
2,Q3,What is AI4AI-Bench designed to evaluate?,AI4AI‑Bench is a benchmark that tests whether ...,2608.20318v1,2,2,2
3,Q4,How does AI4AI-Bench assess the performance of...,AI4AI‑Bench evaluates an LLM agent by letting ...,2608.20318v1,2,2,2
4,Q5,What information is used to induce task models...,Task models are induced from the low‑level eve...,2608.20319v1,2,2,2
5,Q6,What is the purpose of the agentic approach fo...,The agentic approach is designed to unify the ...,2608.20320v1,2,2,2
6,Q7,What problem does grounded checklist-aligned r...,Grounded checklist‑aligned reward learning (G‑...,2608.20331v1,2,2,2
7,Q8,How is reward learning used to improve patient...,Reward learning in G‑CARL improves patient‑ori...,2608.20331v1,2,2,2


In [41]:
# Calculate generation metrics

answer_relevance = (
    answer_evaluation_df["answer_relevance"].mean() / 2
)

faithfulness = (
    answer_evaluation_df["faithfulness"].mean() / 2
)

citation_accuracy = (
    answer_evaluation_df["citation_accuracy"].mean() / 2
)

print("Baseline RAG Generation Performance")
print("-" * 40)
print(f"Answer Relevance: {answer_relevance:.2%}")
print(f"Faithfulness: {faithfulness:.2%}")
print(f"Citation Accuracy: {citation_accuracy:.2%}")

Baseline RAG Generation Performance
----------------------------------------
Answer Relevance: 100.00%
Faithfulness: 100.00%
Citation Accuracy: 100.00%


In [42]:
# Final baseline RAG results

final_results = pd.DataFrame({
    "Metric": [
        "Hit@1",
        "Hit@3",
        "Hit@5",
        "Answer Relevance",
        "Faithfulness",
        "Citation Accuracy"
    ],
    "Score": [
        hit_at_1,
        hit_at_3,
        hit_at_5,
        answer_relevance,
        faithfulness,
        citation_accuracy
    ]
})

final_results["Score"] = final_results["Score"].map(
    lambda x: f"{x:.2%}"
)

final_results

,Metric,Score
0,Hit@1,100.00%
1,Hit@3,100.00%
2,Hit@5,100.00%
3,Answer Relevance,100.00%
4,Faithfulness,100.00%
5,Citation Accuracy,100.00%


In [43]:
# Save generated RAG answers

rag_results_df.to_csv(
    processed_dir / "baseline_rag_answers.csv",
    index=False
)

print("Saved: baseline_rag_answers.csv")

Saved: baseline_rag_answers.csv


In [44]:
# Save manual answer evaluation

answer_evaluation_df.to_csv(
    processed_dir / "baseline_answer_evaluation.csv",
    index=False
)

print("Saved: baseline_answer_evaluation.csv")

Saved: baseline_answer_evaluation.csv


In [45]:
# Save final baseline RAG results

final_results.to_csv(
    processed_dir / "baseline_rag_results.csv",
    index=False
)

print("Saved: baseline_rag_results.csv")

Saved: baseline_rag_results.csv


## Baseline RAG Evaluation Summary

The baseline RAG pipeline was evaluated using an eight-question benchmark covering the five research papers in the dataset. Retrieval performance was measured using Hit@1, Hit@3, and Hit@5, while generated responses were manually evaluated for answer relevance, faithfulness to the retrieved context, and citation accuracy.

The baseline achieved 100% Hit@1, Hit@3, and Hit@5 on the benchmark. The generated responses also achieved 100% on answer relevance, faithfulness, and citation accuracy based on the manual evaluation.

These results indicate that the baseline pipeline successfully retrieved the expected source paper and generated relevant, context-supported answers for the benchmark questions. However, the benchmark contains only eight questions, so these results should be interpreted as baseline performance on the current evaluation set rather than evidence of general RAG performance.

## Evaluation Methodology

### Retrieval Evaluation

Retrieval performance was evaluated using three metrics:

- **Hit@1:** whether the expected source paper was ranked first.
- **Hit@3:** whether the expected source paper appeared within the top three retrieved results.
- **Hit@5:** whether the expected source paper appeared within the top five retrieved results.

### Generation Evaluation

Generated answers were manually evaluated using a 0–2 scale:

- **Answer Relevance:** whether the response directly answers the benchmark question.
- **Faithfulness:** whether the response is supported by the retrieved research context.
- **Citation Accuracy:** whether the cited arXiv source corresponds to the expected source paper.

A score of 2 represents a strong result, 1 represents a partial result, and 0 represents an incorrect or unsupported result.